In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, Dataset

In [ ]:
class LSTConvNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.conv1d = nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            padding=kernel_size - 1,
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.conv1d(X))
        out = out[:, :, :-(self.kernel_size - 1)].contiguous()
        out = self.dropout(out)
        return out
    

class LSTGruNet(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, batch_first=True)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(X)
        return self.dropout(out)


class LSTSkipGruNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_step_sizes: list[int] | None = None,
        skip_out_channels: list[int] | None = None,
        dropout: float = 0.2,
    ):
        super().__init__()
        
        self.rnn_skip_step_sizes = skip_step_sizes
        self.rnn_skip_out_channels = skip_out_channels
        self.rnn_skip_nets = nn.ModuleList()
        for i in range(len(self.rnn_skip_step_sizes)):
            skip_net = nn.GRU(
                input_size=in_channels,
                hidden_size=skip_out_channels[i],
                batch_first=True
            )
            self.rnn_skip_nets.append(skip_net)
        self.skip_dropout = nn.Dropout(dropout)

    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        skip_outputs: list[torch.Tensor] = []
        
        batch_size, _, n_timesteps = X.size()  # assume X is [batch_size, in_features, timesteps]
        for i, skip_step_size in enumerate(self.rnn_skip_step_sizes):
            n_skip_sequences = n_timesteps // skip_step_size

            # Only keep n_skips * skip_step_size worth of skips
            # [batch_size, cnn_out_channels, n_skips*skip_step_size]
            S = X[:, :, -n_skip_sequences*skip_step_size:].contiguous()

            # Reshape last time axis into maps of length skip_step_size so the time axis
            # is now a matrix of n_skips (rows) each of length skip_step_size (columns)
            S = S.view(S.size(0), S.size(1), n_skip_sequences, skip_step_size)

            # Permute to [batch_size, skip_step_size, n_skips, conv_out_channels]
            S = S.permute(0, 3, 2, 1).contiguous()

            # Collapse first batch_size and skip_step_size dimensions into single dimension
            # [bath_size * skip_step_size, n_skips, conv_out_channels]
            S = S.view(S.size(0) * S.size(1), S.size(2), S.size(3))

            # Apply GRU for this skip step size
            S_out, _ = self.rnn_skip_nets[i](S)
            
            # Keep the outputs of the final skip step
            # These are the final embeddings for all "phases" within the given 
            # skip step size e.g. for skip step size of 5 there will be 5 embeddings
            # one for each (1, 2, 3, 4, 5) phase
            # Shape is [batch_size * skip_step, skip_out_channels]
            S_out = S_out[:, -1, :]
    
            # Bring back the batch dimension
            S_out = S_out.view(batch_size, skip_step_size * S_out.size(1))
            S_out = self.skip_dropout(S_out)
            skip_outputs.append(S_out)
        
        return torch.cat(skip_outputs, dim=1)


class LSTNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        cnn_out_channels: int = 32,
        cnn_kernel_size: int = 2,
        rnn_out_channels: int = 64,
        rnn_skip_step_sizes: list[int] | None = None,
        rnn_skip_out_channels: list[int] | None = None,
        ar_window_size: int = 0,
        dropout: float = 0.2
    ):
        super().__init__()

        self.conv_net = LSTConvNet(
            in_channels=in_channels,
            out_channels=cnn_out_channels,
            kernel_size=cnn_kernel_size,
            dropout=dropout,
        )
        
        # Recurrent GRU Net
        self.rnn_net = LSTGruNet(
            input_size=cnn_out_channels,
            hidden_size=rnn_out_channels,
            dropout=dropout,
        )

        # Recurrent GRU Skip Nets
        assert len(rnn_skip_step_sizes) == len(rnn_skip_out_channels)
        self.skip_rnn_net = LSTSkipGruNet(
            in_channels=cnn_out_channels,
            skip_step_sizes=rnn_skip_step_sizes,
            skip_out_channels=rnn_skip_out_channels,
            dropout=dropout
        )
        
        # Linear layer to decode rnn + skip rnn outputs
        decoder_in_channels = rnn_out_channels + np.dot(rnn_skip_step_sizes, rnn_skip_out_channels)
        self.decoder_net = nn.Linear(in_features=decoder_in_channels, out_features=out_channels)
        
        # Highway Net
        self.target_index = -1
        self.ar_window_size = ar_window_size
        if self.ar_window_size > 0:
            self.ar_net = nn.Linear(self.ar_window_size, out_features=out_channels)
        


    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Conv net
        conv_in = X.permute(0, 2, 1).contiguous()  # [batch_size, in_channels, time_steps] 
        conv_out = self.conv_net(conv_in)          # [batch_size, cnn_out_channels, time_steps]

        # Recurrent GRU Net
        gru_in = conv_out.permute(0, 2, 1)       # [batch_size, time_steps, cnn_out_channels]
        gru_out = self.rnn_net(gru_in)           # [batch_size, time_steps, rnn_out_channels]
        gru_out = gru_out[:, -1, :]              # [batch_size, rnn_out_channels]

        # Recurrent skip GRUs
        skip_out = self.skip_rnn_net(conv_out)           # [batch_size, dot(rnn_skip_step_sizes, rnn_skip_out_channels)]
        decoder_in = torch.cat((gru_out, skip_out), 1)
        
        # Output decoder layer
        decoder_out = self.decoder_net(decoder_in)  # [batch_size, out_channels]
        
        # Ar net
        if self.ar_window_size > 0:
            AR = X[:, -self.ar_window_size:, [self.target_index]]  # [batch_size, ar_window, 1]
            AR = AR.permute(0, 2, 1)                               # [batch_size, 1, ar_window]
            AR = self.ar_net(AR).squeeze(1)                        # [batch_size, out_channels]
            decoder_out = decoder_out + AR

        return decoder_out

In [ ]:
n_timesteps = 100
period = 24
t = np.arange(n_timesteps)
y = np.sin(2 * np.pi * t / period)

plt.plot(y)

In [ ]:
# Transform into 
input_seq_length = 10
out_seq_length = 10

n_samples = len(y) - input_seq_length - out_seq_length + 1

features = np.empty((n_samples, input_seq_length, 1), dtype=np.float32)
labels = np.empty((n_samples, out_seq_length, 1), dtype=np.float32)

for i in range(n_samples):
    feat_start, feat_end = i, i + input_seq_length
    features[i] = y[feat_start: feat_end].reshape(-1, 1)

    labels_start, labels_end = feat_end, feat_end + out_seq_length
    labels[i] = y[labels_start: labels_end].reshape(-1, 1)

features_ts = torch.from_numpy(features)
labels_ts = torch.from_numpy(labels)

train_ds = TensorDataset(features_ts, labels_ts)
train_dl = DataLoader(train_ds, batch_size=32)

In [ ]:
model = LSTNet(
    in_channels=1,
    out_channels=10,
    cnn_out_channels=5,
    cnn_kernel_size=2,
    rnn_out_channels=7,
    rnn_skip_step_sizes=[5],
    rnn_skip_out_channels=[4],
    ar_window_size=5
)

loss_fn = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-03)

model.train()
epoch_loss, batch_loss = [], []
n_epochs = 200
epoch_pgbar = tqdm(range(n_epochs))
for epoch in epoch_pgbar:
    for batch_X, batch_y in train_dl:
        optimizer.zero_grad()
        
        y_hat = model(batch_X)

        # Output dimension is (batch_size, out_seq_length)
        y_hat = y_hat.unsqueeze(-1)
        loss = loss_fn(batch_y, y_hat)
        loss.backward()
        optimizer.step()
        
        loss_detach = float(loss.detach())
        batch_loss.append(loss_detach)
    
    epoch_loss.append(loss_detach)
    epoch_pgbar.set_description(f"Epoch [{epoch + 1} / {n_epochs}] - Loss = {loss_detach:.3f}")

In [ ]:
test_idx = np.random.choice(features_ts.size(0))

test_X = features_ts[[test_idx]]
test_y = labels_ts[[test_idx]]

model.eval()
with torch.no_grad():
    y_hat = model(test_X)

plt.plot(
    np.arange(test_X.size(1)),
    test_X.squeeze().numpy(),
    label="train",
)
plt.plot(
    np.arange(test_X.size(1), test_X.size(1) + test_y.size(1)),
    test_y.squeeze().numpy(),
    label="test",
)
plt.plot(
    np.arange(test_X.size(1), test_X.size(1) + test_y.size(1)),
    y_hat.squeeze().numpy(),
    label="predicted"
)

plt.legend()